In [12]:
import pandas as pd

# Load RAW files
vendors = pd.read_csv("../data/raw/vendors.csv")
products = pd.read_csv("../data/raw/products.csv")
orders = pd.read_csv("../data/raw/purchase_orders.csv")
deliveries = pd.read_csv("../data/raw/deliveries.csv")

In [13]:
def missing_report(df, name):
    miss = df.isna().sum()          # count of missing values per column
    miss = miss[miss > 0]           # keep only columns that have missing values
    print(f"--- {name} ---")
    if miss.empty:
        print("No missing values")
    else:
        print(pd.DataFrame({"missing": miss,
                            "missing_%": (miss / len(df) * 100).round(2)}))

for name, df in [("vendors", vendors), ("products", products),
                 ("orders", orders), ("deliveries", deliveries)]:
    missing_report(df, name)

--- vendors ---
      missing  missing_%
city        2        8.0
--- products ---
No missing values
--- orders ---
            missing  missing_%
unit_price       40       0.66
--- deliveries ---
                      missing  missing_%
actual_delivery_date       20       0.34


In [14]:
print("Duplicate rows (orders):", orders.duplicated().sum())
orders = orders.drop_duplicates()
print("Shape after:", orders.shape)
print("po_id unique?", orders["po_id"].is_unique)

# Check the other tables too
for name, df in [("vendors", vendors), ("products", products), ("deliveries", deliveries)]:
    print(name, "duplicates:", df.duplicated().sum())

Duplicate rows (orders): 60
Shape after: (6000, 9)
po_id unique? True
vendors duplicates: 0
products duplicates: 0
deliveries duplicates: 0


In [15]:
mask = orders["unit_price"].isna()                      # True for the 40 missing rows

# Safety check: formula tabhi sahi hai jab quantity valid ho
print((orders.loc[mask, "quantity_ordered"] > 0).all())  # expect True

orders.loc[mask, "unit_price"] = (
    orders.loc[mask, "total_amount"] / orders.loc[mask, "quantity_ordered"]
).round(2)

print("Missing unit_price now:", orders["unit_price"].isna().sum())
orders.loc[mask].head(3)[["po_id", "quantity_ordered", "unit_price", "total_amount"]]

True
Missing unit_price now: 0


,po_id,quantity_ordered,unit_price,total_amount
308,PO004517,69,144.53,9972.57
401,PO005835,148,508.47,75253.56
431,PO002379,124,1072.04,132932.96


In [16]:
vendors["city"] = vendors["city"].fillna("Unknown")
print(vendors["city"].isna().sum())     # expect 0

0


In [17]:
deliveries["delivery_date_missing"] = deliveries["actual_delivery_date"].isna()
print(deliveries["delivery_date_missing"].sum())    # expect 20

20


In [18]:
for name, df in [("vendors", vendors), ("products", products),
                 ("orders", orders), ("deliveries", deliveries)]:
    missing_report(df, name)

--- vendors ---
No missing values
--- products ---
No missing values
--- orders ---
No missing values
--- deliveries ---
                      missing  missing_%
actual_delivery_date       20       0.34


In [19]:
print(orders.dtypes)
print(deliveries.dtypes)
print(vendors.dtypes)

po_id                         str
vendor_id                     str
product_id                    str
order_date                    str
expected_delivery_date        str
quantity_ordered            int64
unit_price                float64
total_amount              float64
po_status                     str
dtype: object
delivery_id                str
po_id                      str
actual_delivery_date       str
quantity_received        int64
quantity_defective       int64
delivery_date_missing     bool
dtype: object
vendor_id                str
vendor_name              str
category                 str
city                     str
payment_terms_days     int64
contract_start_date      str
status                   str
dtype: object


In [20]:
iso_pattern = r"^\d{4}-\d{2}-\d{2}$"      # matches YYYY-MM-DD

for col in ["order_date", "expected_delivery_date"]:
    not_iso = ~orders[col].str.match(iso_pattern)
    print(col, "->", not_iso.sum(), "rows are not YYYY-MM-DD")
    print(orders.loc[not_iso, col].head(3).tolist())

print(deliveries["actual_delivery_date"].dropna().str.match(iso_pattern).all())
print(vendors["contract_start_date"].str.match(iso_pattern).all())

order_date -> 467 rows are not YYYY-MM-DD
['26-05-2024', '05-01-2024', '20-04-2024']
expected_delivery_date -> 476 rows are not YYYY-MM-DD
['13-12-2025', '12-08-2025', '30-07-2024']
True
True


In [22]:
def parse_mixed_dates(series):
    iso = pd.to_datetime(series, format="%Y-%m-%d", errors="coerce")   # parse YYYY-MM-DD
    dmy = pd.to_datetime(series, format="%d-%m-%Y", errors="coerce")   # parse DD-MM-YYYY
    return iso.fillna(dmy)                                             # take ISO result, else DMY

orders["order_date"] = parse_mixed_dates(orders["order_date"])
orders["expected_delivery_date"] = parse_mixed_dates(orders["expected_delivery_date"])
deliveries["actual_delivery_date"] = parse_mixed_dates(deliveries["actual_delivery_date"])
vendors["contract_start_date"] = parse_mixed_dates(vendors["contract_start_date"])

In [23]:
# 1) Did any date fail to parse?
print(orders[["order_date", "expected_delivery_date"]].isna().sum())
print("Missing delivery dates:", deliveries["actual_delivery_date"].isna().sum())
print("Missing contract dates:", vendors["contract_start_date"].isna().sum())

# 2) New data types
print(orders[["order_date", "expected_delivery_date"]].dtypes)

# 3) Are the date ranges realistic?
print(orders["order_date"].min(), "->", orders["order_date"].max())
print(deliveries["actual_delivery_date"].min(), "->", deliveries["actual_delivery_date"].max())
print(vendors["contract_start_date"].min(), "->", vendors["contract_start_date"].max())

# 4) Logic checks
print("Expected date before order date:",
      (orders["expected_delivery_date"] < orders["order_date"]).sum())

check = deliveries.merge(orders[["po_id", "order_date"]], on="po_id")
print("Delivered before ordered:",
      (check["actual_delivery_date"] < check["order_date"]).sum())

order_date                0
expected_delivery_date    0
dtype: int64
Missing delivery dates: 20
Missing contract dates: 0
order_date                datetime64[us]
expected_delivery_date    datetime64[us]
dtype: object
2024-01-01 00:00:00 -> 2025-12-31 00:00:00
2024-01-04 00:00:00 -> 2026-01-26 00:00:00
2019-03-20 00:00:00 -> 2023-10-08 00:00:00
Expected date before order date: 0
Delivered before ordered: 0
